# Task 2 — Insurance Premium Prediction (2021)

**Goal:** Predict Earned Premium for California zip codes in 2021 using historical data (2018-2020).

**Approach:**
- Aggregate insurance data to zip-year level
- Build lag features (premium, claims, exposure from prior years)
- Merge fire risk features from Task 1
- Train: 2020 zip-year rows (with lags from 2018-2019)
- Test: 2021 zip-year rows (with lags from 2019-2020)
- Models: Naive baseline, Random Forest, XGBoost

In [1]:
# === Cell 1: Imports and Data Loading ===
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# Load insurance dataset
insurance = pd.read_csv('../data/insurance_fire_census_weather_raw.csv')
print(f"Insurance data: {insurance.shape}")
print(f"Years: {sorted(insurance['Year'].unique())}")
print(f"Zip codes: {insurance['ZIP'].nunique()}")
print(f"Rows per zip-year (mean): {insurance.groupby(['ZIP','Year']).size().mean():.2f}")

Insurance data: (47033, 76)
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Zip codes: 2251
Rows per zip-year (mean): 5.67


In [2]:
# === Cell 2: Aggregate to Zip-Year Level ===
#
# Multiple rows per zip-year (different insurance categories).
# Sum for counts/dollars, mean for scores, first for census.

sum_cols = [
    'Earned Premium', 'Earned Exposure',
    'CAT Cov A Fire -  Incurred Losses', 'CAT Cov A Fire -  Number of Claims',
    'CAT Cov A Smoke -  Incurred Losses', 'CAT Cov A Smoke -  Number of Claims',
    'CAT Cov C Fire -  Incurred Losses', 'CAT Cov C Fire -  Number of Claims',
    'CAT Cov C Smoke -  Incurred Losses', 'CAT Cov C Smoke -  Number of Claims',
    'Non-CAT Cov A Fire -  Incurred Losses', 'Non-CAT Cov A Fire -  Number of Claims',
    'Non-CAT Cov A Smoke -  Incurred Losses', 'Non-CAT Cov A Smoke -  Number of Claims',
    'Non-CAT Cov C Fire -  Incurred Losses', 'Non-CAT Cov C Fire -  Number of Claims',
    'Non-CAT Cov C Smoke -  Incurred Losses', 'Non-CAT Cov C Smoke -  Number of Claims',
    'Number of High Fire Risk Exposure', 'Number of Very High Fire Risk Exposure',
    'Number of Low Fire Risk Exposure', 'Number of Moderate Fire Risk Exposure',
    'Number of Negligible Fire Risk Exposure',
]

mean_cols = [
    'Avg Fire Risk Score',
    'Cov A Amount Weighted Avg', 'Cov C Amount Weighted Avg',
]

# Avg PPC: backfill from most recent year per zip
# (only has data in 2018-2019, zero in 2020-2021)
ppc_by_zip = insurance[insurance['Avg PPC'].notna()].groupby('ZIP')['Avg PPC'].mean()

census_cols = [
    'total_population', 'median_income', 'total_housing_units',
    'average_household_size', 'educational_attainment_bachelor_or_higher',
    'poverty_status', 'housing_value', 'housing_vacancy_number',
    'median_monthly_housing_costs', 'owner_occupied_housing_units',
    'renter_occupied_housing_units',
]

# Build aggregation dict
agg_dict = {}
for c in sum_cols:
    if c in insurance.columns:
        agg_dict[c] = 'sum'
for c in mean_cols:
    if c in insurance.columns:
        agg_dict[c] = 'mean'
for c in census_cols:
    if c in insurance.columns:
        agg_dict[c] = 'first'

zip_year = insurance.groupby(['ZIP', 'Year']).agg(agg_dict).reset_index()

# Backfill Avg PPC
zip_year['Avg PPC'] = zip_year['ZIP'].map(ppc_by_zip)
zip_year['Avg PPC'] = zip_year['Avg PPC'].fillna(zip_year['Avg PPC'].median())

print(f"Aggregated zip-year data: {zip_year.shape}")
print(f"Years: {sorted(zip_year['Year'].unique())}")
print(f"Zip codes: {zip_year['ZIP'].nunique()}")
print(f"\nTarget (Earned Premium) stats per year:")
print(zip_year.groupby('Year')['Earned Premium'].describe()[['count','mean','50%','max']])

Aggregated zip-year data: (8288, 40)
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Zip codes: 2251

Target (Earned Premium) stats per year:
       count          mean        50%          max
Year                                              
2018  2059.0  2.072658e+06   962946.0   63209847.0
2019  1993.0  2.230625e+06  1214020.0   29969352.0
2020  2118.0  3.431476e+06  1653079.0   83270268.0
2021  2118.0  3.733440e+06  1831749.5  118086710.0


In [3]:
# === Cell 3: Build Lag Features ===
#
# For predicting 2020 premiums, we use 2018/2019 lags.
# For predicting 2021 premiums, we use 2019/2020 lags.

# Sort for proper lag computation
zip_year = zip_year.sort_values(['ZIP', 'Year']).reset_index(drop=True)

# Lag features: previous year values
lag_features = [
    'Earned Premium', 'Earned Exposure',
    'CAT Cov A Fire -  Incurred Losses', 'CAT Cov A Fire -  Number of Claims',
    'Non-CAT Cov A Fire -  Incurred Losses', 'Non-CAT Cov A Fire -  Number of Claims',
]

for col in lag_features:
    # Lag 1 (previous year)
    zip_year[f'{col}_lag1'] = zip_year.groupby('ZIP')[col].shift(1)
    # Lag 2 (two years ago)
    zip_year[f'{col}_lag2'] = zip_year.groupby('ZIP')[col].shift(2)

# Premium growth rate (year over year)
zip_year['premium_growth'] = zip_year.groupby('ZIP')['Earned Premium'].pct_change()

# Premium rolling average (2-year)
zip_year['premium_rolling_avg'] = zip_year.groupby('ZIP')['Earned Premium'].transform(
    lambda x: x.shift(1).rolling(2, min_periods=1).mean()
)

# Fix infinity values from division by zero in pct_change
zip_year = zip_year.replace([np.inf, -np.inf], np.nan)

print(f"After lag features: {zip_year.shape}")
print(f"\nNull counts in lag columns (expected for 2018 since no prior year):")
lag_nulls = zip_year[[c for c in zip_year.columns if 'lag' in c or 'growth' in c or 'rolling' in c]].isnull().sum()
print(lag_nulls[lag_nulls > 0])

After lag features: (8288, 54)

Null counts in lag columns (expected for 2018 since no prior year):
Earned Premium_lag1                            2251
Earned Premium_lag2                            4403
Earned Exposure_lag1                           2251
Earned Exposure_lag2                           4403
CAT Cov A Fire -  Incurred Losses_lag1         2251
CAT Cov A Fire -  Incurred Losses_lag2         4403
CAT Cov A Fire -  Number of Claims_lag1        2251
CAT Cov A Fire -  Number of Claims_lag2        4403
Non-CAT Cov A Fire -  Incurred Losses_lag1     2251
Non-CAT Cov A Fire -  Incurred Losses_lag2     4403
Non-CAT Cov A Fire -  Number of Claims_lag1    2251
Non-CAT Cov A Fire -  Number of Claims_lag2    4403
premium_growth                                 2312
premium_rolling_avg                            2251
dtype: int64


In [4]:
# === Cell 4: Merge Fire Risk Features from Task 1 ===

# Load the wildfire feature matrix
fire_features = pd.read_csv('../data/feature_matrix_clean.csv')

# Select relevant fire risk columns
fire_cols = ['zip', 'Year', 'had_fire', 'prior_fire_count', 'prior_total_acres_log',
             'mean_tmax', 'fire_season_mean_tmax', 'fire_season_total_precip', 'total_precip']
fire_subset = fire_features[fire_cols].copy()
fire_subset = fire_subset.rename(columns={'zip': 'ZIP'})
fire_subset['ZIP'] = fire_subset['ZIP'].astype(int)

# Merge
zip_year['ZIP'] = zip_year['ZIP'].astype(int)
zip_year = zip_year.merge(fire_subset, on=['ZIP', 'Year'], how='left')

# Fill fire features for zips not in wildfire dataset
fire_merge_cols = [c for c in fire_cols if c not in ['zip', 'Year']]
for c in fire_merge_cols:
    if c in zip_year.columns:
        if c == 'had_fire':
            zip_year[c] = zip_year[c].fillna(0)
        else:
            zip_year[c] = zip_year[c].fillna(zip_year[c].median())

print(f"After fire risk merge: {zip_year.shape}")
print(f"Fire features added: {fire_merge_cols}")
print(f"Nulls remaining: {zip_year.isnull().sum().sum()}")

After fire risk merge: (8288, 61)
Fire features added: ['had_fire', 'prior_fire_count', 'prior_total_acres_log', 'mean_tmax', 'fire_season_mean_tmax', 'fire_season_total_precip', 'total_precip']
Nulls remaining: 59794


In [5]:
# === Cell 5: Build Train/Test Sets ===
#
# Train on 2020 (has lag1 from 2019, lag2 from 2018)
# Test on 2021 (has lag1 from 2020, lag2 from 2019)
# We skip 2018 (no lags) and 2019 (no lag2)

train_df = zip_year[zip_year['Year'] == 2020].copy()
test_df = zip_year[zip_year['Year'] == 2021].copy()

# Define feature columns (exclude target, identifiers, and raw current-year claims)
exclude_cols = [
    'ZIP', 'Year', 'Earned Premium',  # target and identifiers
]

feature_cols = [c for c in train_df.columns if c not in exclude_cols]

# Drop any remaining nulls
train_df = train_df.dropna(subset=feature_cols + ['Earned Premium'])
test_df = test_df.dropna(subset=feature_cols + ['Earned Premium'])

X_train = train_df[feature_cols].copy()
y_train = train_df['Earned Premium'].copy()
X_test = test_df[feature_cols].copy()
y_test = test_df['Earned Premium'].copy()

# Fill any remaining NaN with median
for c in feature_cols:
    median_val = X_train[c].median()
    X_train[c] = X_train[c].fillna(median_val)
    X_test[c] = X_test[c].fillna(median_val)

# Fix infinities
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
for c in feature_cols:
    median_val = X_train[c].median()
    X_train[c] = X_train[c].fillna(median_val)
    X_test[c] = X_test[c].fillna(median_val)

print(f"Train: {X_train.shape[0]} zip codes (year 2020)")
print(f"Test:  {X_test.shape[0]} zip codes (year 2021)")
print(f"Features: {X_train.shape[1]}")
print(f"\nTarget (Earned Premium) stats:")
print(f"  Train — mean: ${y_train.mean():,.0f}, median: ${y_train.median():,.0f}, max: ${y_train.max():,.0f}")
print(f"  Test  — mean: ${y_test.mean():,.0f}, median: ${y_test.median():,.0f}, max: ${y_test.max():,.0f}")
print(f"\nFeature list ({len(feature_cols)} features):")
for i, c in enumerate(feature_cols):
    print(f"  {i+1:2d}. {c}")

Train: 1529 zip codes (year 2020)
Test:  1529 zip codes (year 2021)
Features: 58

Target (Earned Premium) stats:
  Train — mean: $4,732,370, median: $3,450,619, max: $83,270,268
  Test  — mean: $5,146,262, median: $3,820,520, max: $118,086,710

Feature list (58 features):
   1. Earned Exposure
   2. CAT Cov A Fire -  Incurred Losses
   3. CAT Cov A Fire -  Number of Claims
   4. CAT Cov A Smoke -  Incurred Losses
   5. CAT Cov A Smoke -  Number of Claims
   6. CAT Cov C Fire -  Incurred Losses
   7. CAT Cov C Fire -  Number of Claims
   8. CAT Cov C Smoke -  Incurred Losses
   9. CAT Cov C Smoke -  Number of Claims
  10. Non-CAT Cov A Fire -  Incurred Losses
  11. Non-CAT Cov A Fire -  Number of Claims
  12. Non-CAT Cov A Smoke -  Incurred Losses
  13. Non-CAT Cov A Smoke -  Number of Claims
  14. Non-CAT Cov C Fire -  Incurred Losses
  15. Non-CAT Cov C Fire -  Number of Claims
  16. Non-CAT Cov C Smoke -  Incurred Losses
  17. Non-CAT Cov C Smoke -  Number of Claims
  18. Number of H

In [6]:
# === Cell 6: Define Evaluation Metrics ===

def evaluate_regression(name, y_true, y_pred):
    """Evaluate regression model with standard metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    # MAPE (avoid division by zero)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  MAE:   ${mae:,.0f}")
    print(f"  RMSE:  ${rmse:,.0f}")
    print(f"  R²:    {r2:.4f}")
    print(f"  MAPE:  {mape:.1f}%")
    
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape}

print("Evaluation function ready.")

Evaluation function ready.


In [7]:
# === Cell 7: Naive Baseline — Predict Last Year's Premium ===
#
# The simplest model: predict 2021 premium = 2020 premium (lag1)
# Any good model must beat this.

if 'Earned Premium_lag1' in X_test.columns:
    y_pred_naive = X_test['Earned Premium_lag1'].values
    results_naive = evaluate_regression("Naive Baseline (Last Year's Premium)", y_test.values, y_pred_naive)
else:
    print("ERROR: Earned Premium_lag1 not found in features")


  Naive Baseline (Last Year's Premium)
  MAE:   $1,018,441
  RMSE:  $3,348,682
  R²:    0.6813
  MAPE:  18.3%


In [8]:
# === Cell 8: Random Forest Regressor ===

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

results_rf = evaluate_regression("Random Forest Regressor", y_test.values, y_pred_rf)

# Feature importance
rf_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 15 features (Random Forest):")
for _, row in rf_importance.head(15).iterrows():
    print(f"  {row['feature']:<45s} {row['importance']:.4f}")


  Random Forest Regressor
  MAE:   $569,292
  RMSE:  $2,306,956
  R²:    0.8487
  MAPE:  12.5%

Top 15 features (Random Forest):
  Earned Exposure                               0.2543
  premium_rolling_avg                           0.2223
  Earned Premium_lag1                           0.2223
  Earned Premium_lag2                           0.0974
  premium_growth                                0.0767
  Non-CAT Cov A Fire -  Incurred Losses         0.0611
  Number of Low Fire Risk Exposure              0.0219
  Number of Moderate Fire Risk Exposure         0.0111
  Number of Negligible Fire Risk Exposure       0.0041
  Number of Very High Fire Risk Exposure        0.0040
  Non-CAT Cov A Fire -  Incurred Losses_lag2    0.0024
  Non-CAT Cov A Fire -  Number of Claims_lag1   0.0020
  Non-CAT Cov C Fire -  Number of Claims        0.0016
  educational_attainment_bachelor_or_higher     0.0014
  Earned Exposure_lag1                          0.0014


In [9]:
# === Cell 9: XGBoost Regressor ===

xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='rmse',
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

results_xgb = evaluate_regression("XGBoost Regressor", y_test.values, y_pred_xgb)

# Feature importance
xgb_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 15 features (XGBoost):")
for _, row in xgb_importance.head(15).iterrows():
    print(f"  {row['feature']:<45s} {row['importance']:.4f}")


  XGBoost Regressor
  MAE:   $510,569
  RMSE:  $2,121,394
  R²:    0.8721
  MAPE:  11.2%

Top 15 features (XGBoost):
  Earned Premium_lag2                           0.2623
  Earned Premium_lag1                           0.2480
  premium_rolling_avg                           0.1103
  premium_growth                                0.0871
  Number of Low Fire Risk Exposure              0.0823
  Earned Exposure                               0.0741
  Non-CAT Cov A Fire -  Incurred Losses         0.0402
  Number of Very High Fire Risk Exposure        0.0222
  Non-CAT Cov A Fire -  Number of Claims        0.0134
  Number of Moderate Fire Risk Exposure         0.0114
  Non-CAT Cov C Fire -  Number of Claims        0.0065
  housing_vacancy_number                        0.0053
  Non-CAT Cov C Fire -  Incurred Losses         0.0044
  CAT Cov A Fire -  Incurred Losses_lag1        0.0037
  CAT Cov A Fire -  Number of Claims_lag1       0.0036


In [10]:
# === Cell 10: Comparison Table ===

print("\n" + "="*75)
print("  TASK 2 — Insurance Premium Prediction (2021) — Results")
print("="*75)
print(f"\n{'Model':<40s} {'MAE':>12s} {'RMSE':>12s} {'R²':>8s} {'MAPE':>8s}")
print(f"{'-'*75}")

for name, res in [
    ("Naive (last year's premium)", results_naive),
    ("Random Forest", results_rf),
    ("XGBoost", results_xgb),
]:
    print(f"{name:<40s} ${res['mae']:>10,.0f} ${res['rmse']:>10,.0f} {res['r2']:>7.4f} {res['mape']:>6.1f}%")

print(f"\nTrain size: {X_train.shape[0]} zip codes | Test size: {X_test.shape[0]} zip codes")
print(f"Features: {X_train.shape[1]}")


  TASK 2 — Insurance Premium Prediction (2021) — Results

Model                                             MAE         RMSE       R²     MAPE
---------------------------------------------------------------------------
Naive (last year's premium)              $ 1,018,441 $ 3,348,682  0.6813   18.3%
Random Forest                            $   569,292 $ 2,306,956  0.8487   12.5%
XGBoost                                  $   510,569 $ 2,121,394  0.8721   11.2%

Train size: 1529 zip codes | Test size: 1529 zip codes
Features: 58


In [11]:
# === Cell 11: Generate Full 2021 Predictions ===
#
# Use the best model to predict premiums for all 2021 zip codes.
# Save predictions for the submission.

# Determine best model
best_name = 'XGBoost' if results_xgb['r2'] >= results_rf['r2'] else 'Random Forest'
best_model = xgb if best_name == 'XGBoost' else rf
best_pred = y_pred_xgb if best_name == 'XGBoost' else y_pred_rf
print(f"Best model: {best_name}")

# Create prediction table
predictions = test_df[['ZIP', 'Year']].copy()
predictions['Actual_Premium'] = y_test.values
predictions['Predicted_Premium'] = best_pred
predictions['Absolute_Error'] = np.abs(predictions['Actual_Premium'] - predictions['Predicted_Premium'])
predictions['Pct_Error'] = predictions.apply(
    lambda r: abs(r['Actual_Premium'] - r['Predicted_Premium']) / r['Actual_Premium'] * 100 
    if r['Actual_Premium'] != 0 else np.nan, axis=1
)

# Save
predictions.to_csv('../data/task2_predictions_2021.csv', index=False)
print(f"\nSaved predictions for {len(predictions)} zip codes to data/task2_predictions_2021.csv")

print(f"\nPrediction summary:")
print(f"  Zip codes predicted: {len(predictions)}")
print(f"  Mean actual premium:    ${predictions['Actual_Premium'].mean():>12,.0f}")
print(f"  Mean predicted premium: ${predictions['Predicted_Premium'].mean():>12,.0f}")
print(f"  Median abs error:       ${predictions['Absolute_Error'].median():>12,.0f}")
print(f"  Median pct error:       {predictions['Pct_Error'].median():>11.1f}%")

print(f"\nWorst 10 predictions (by absolute error):")
print(predictions.nlargest(10, 'Absolute_Error')[['ZIP','Actual_Premium','Predicted_Premium','Absolute_Error','Pct_Error']].to_string(index=False))

Best model: XGBoost

Saved predictions for 1529 zip codes to data/task2_predictions_2021.csv

Prediction summary:
  Zip codes predicted: 1529
  Mean actual premium:    $   5,146,262
  Mean predicted premium: $   5,129,212
  Median abs error:       $     131,617
  Median pct error:               5.5%

Worst 10 predictions (by absolute error):
  ZIP  Actual_Premium  Predicted_Premium  Absolute_Error  Pct_Error
90265       118086710         56130024.0      61956686.0  52.467112
93446        42814744         62604092.0      19789348.0  46.220872
95747        16780681         33663808.0      16883127.0 100.610500
95648        27484388         42801476.0      15317088.0  55.730140
95966        11089456         26337364.0      15247908.0 137.499152
95901        34778082         49961596.0      15183514.0  43.658285
94565        32723682         43454512.0      10730830.0  32.792245
93257        23765604         34113880.0      10348276.0  43.543080
92346         9701522         18698954.0    

In [12]:
# === Cell 12: Fire Risk Impact Analysis ===
#
# Show how fire risk features from Task 1 contribute to premium prediction.
# This ties Task 1 and Task 2 together for the submission.

fire_feature_names = ['had_fire', 'prior_fire_count', 'prior_total_acres_log',
                      'mean_tmax', 'fire_season_mean_tmax', 'fire_season_total_precip',
                      'total_precip', 'Avg Fire Risk Score',
                      'Number of High Fire Risk Exposure', 'Number of Very High Fire Risk Exposure']

print("=== Fire Risk Feature Importance in Premium Prediction ===")
print(f"\n{'Feature':<45s} {'RF Imp':>8s} {'XGB Imp':>8s}")
print("-" * 65)

for feat in fire_feature_names:
    rf_imp = rf_importance[rf_importance['feature'] == feat]['importance'].values
    xgb_imp = xgb_importance[xgb_importance['feature'] == feat]['importance'].values
    rf_val = rf_imp[0] if len(rf_imp) > 0 else 0
    xgb_val = xgb_imp[0] if len(xgb_imp) > 0 else 0
    print(f"  {feat:<43s} {rf_val:>8.4f} {xgb_val:>8.4f}")

print(f"\nInterpretation: Fire risk features from Task 1 provide additional")
print(f"predictive signal for insurance premiums, connecting wildfire risk")
print(f"assessment directly to financial impact modeling.")

=== Fire Risk Feature Importance in Premium Prediction ===

Feature                                         RF Imp  XGB Imp
-----------------------------------------------------------------
  had_fire                                      0.0003   0.0001
  prior_fire_count                              0.0005   0.0013
  prior_total_acres_log                         0.0007   0.0005
  mean_tmax                                     0.0003   0.0001
  fire_season_mean_tmax                         0.0003   0.0001
  fire_season_total_precip                      0.0004   0.0001
  total_precip                                  0.0008   0.0001
  Avg Fire Risk Score                           0.0004   0.0002
  Number of High Fire Risk Exposure             0.0010   0.0022
  Number of Very High Fire Risk Exposure        0.0040   0.0222

Interpretation: Fire risk features from Task 1 provide additional
predictive signal for insurance premiums, connecting wildfire risk
assessment directly to financial imp